In [0]:
select * from customers

In [0]:
describe customers

In [0]:
select customer_id,email, profile:first_name, profile:last_name, profile:address:country from customers

In [0]:
select customer_id,email,from_json(profile) from customers

In [0]:
select profile from customers limit 1

In [0]:
CREATE OR REPLACE TEMP VIEW customers_parsed 
AS  SELECT customer_id,email,from_json(profile,schema_of_json('{"first_name":"Dniren","last_name":"Abby","gender":"Female","address":{"street":"768 Mesta Terrace","city":"Annecy","country":"France"}}')) as profile_struct from customers

In [0]:
select * from customers_parsed


In [0]:
describe customers_parsed

In [0]:
SELECT customer_id, profile_struct.first_name, profile_struct.address.country
FROM customers_parsed

In [0]:
select customer_id,profile_struct.* from customers_parsed

In [0]:
select order_id,books from orders

In [0]:
select order_id,explode(books) from orders order by order_id



In [0]:
select customer_id, collect_set(order_id), 
collect_set(books.book_id) from orders
group by customer_id

In [0]:
SELECT customer_id,
  collect_set(books.book_id) As before_flatten,
  array_distinct(flatten(collect_set(books.book_id))) AS after_flatten
FROM orders
GROUP BY customer_id

In [0]:
CREATE TABLE books
SELECT * FROM read_files("dbfs:/Volumes/dbx_catalog/dbx_schema/bookstore/raw/books-csv/",
format => 'csv',
delimiter =>';',
header =>'true'
);

In [0]:
CREATE OR REPLACE VIEW orders_enriched AS
SELECT *
FROM (
  SELECT *, explode(books) AS book 
  FROM orders) o
INNER JOIN books b
ON o.book.book_id = b.book_id;

In [0]:
CREATE OR REPLACE TABLE transactions AS

SELECT * FROM (
  SELECT
    customer_id,
    book.book_id AS book_id,
    book.quantity AS quantity
  FROM orders_enriched
) PIVOT (
  sum(quantity) FOR book_id in (
    'B01', 'B02', 'B03', 'B04', 'B05', 'B06',
    'B07', 'B08', 'B09', 'B10', 'B11', 'B12'
  )
);

SELECT * FROM transactions

In [0]:
SELECT customer_id,
       profile:first_name,
       profile:address:country
FROM customers;

In [0]:
select 
customer_id,
get_json_object(profile,'$.first_name' ) as first_name,
get_json_object(profile,'$.last_name') as last_name,
get_json_object(profile,'$.address.country') as country
FROM customers